# Spatial spike heatmaps for TunED A/B cells

This notebook:
- takes the same `A_only` / `B_only` tuned cell IDs,
- loads per-session spike+tracking data (`*_video_spike_count_df.parquet`),
- computes per-cell spatial firing-rate heatmaps (spikes/s),
- saves many cells per figure (grid pages) to keep plot counts small.

> Note: update the path config in the next cell to match your data locations.


In [12]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("notebook")


In [13]:
# -------------------------
# Paths / settings
# -------------------------

# Input tuned IDs table (same A/B cells)
TUNED_IDS_LONG = Path(r"/ceph/branco/Jasmine_Laurence/rayleigh_analysis/Top2_TunED/tuned_ids_A_or_B_long.csv")

# Root that contains session processed folders (searched recursively)
PROCESSED_SEARCH_ROOT = Path(r"/ceph/branco/Jasmine_Laurence/Experimental_Data")

# Output folder for figures
OUT_DIR = Path(r"/ceph/branco/Jasmine_Laurence/rayleigh_analysis/Top2_TunED/spatial_spike_heatmaps_tuned_cells")

# Plot controls
NBINS = 30
NROWS = 4
NCOLS = 5
MAX_CELLS_PER_PAGE = NROWS * NCOLS
DPI = 220

# Condition controls
# If True, only use rows that match each tuned row's condition string.
USE_CONDITION_FILTER = True

# Exclusion controls (if columns exist)
EXCLUDE_ESCAPE = True
EXCLUDE_HOMINGS = True
REQUIRE_OUT_OF_SHELTER = True

# Optional cap per tuned label, to keep runtime manageable while testing.
# Set to None for all cells.
MAX_CELLS_PER_LABEL = None


In [14]:
def load_tuned_targets(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Tuned ID table not found: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"session", "cluster_id", "condition", "A_only", "B_only"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns in tuned IDs table: {sorted(missing)}")

    out = df.copy()
    out["A_only"] = out["A_only"].astype(bool)
    out["B_only"] = out["B_only"].astype(bool)
    out = out[out["A_only"] | out["B_only"]].copy()
    out["tuned_label"] = np.where(out["A_only"], "A_only", "B_only")
    out["cluster_id"] = pd.to_numeric(out["cluster_id"], errors="coerce")
    out = out.dropna(subset=["cluster_id"])
    out["cluster_id"] = out["cluster_id"].astype(int)

    cols = ["session", "cluster_id", "condition", "tuned_label"]
    out = out[cols].drop_duplicates().reset_index(drop=True)

    if MAX_CELLS_PER_LABEL is not None:
        out = (
            out.sort_values(["tuned_label", "session", "cluster_id"]) 
               .groupby("tuned_label", group_keys=False)
               .head(int(MAX_CELLS_PER_LABEL))
               .reset_index(drop=True)
        )

    return out


def find_session_parquet_files(root: Path) -> dict:
    if not root.exists():
        raise FileNotFoundError(f"Processed root not found: {root}")

    # Matches e.g. good_video_spike_count_df.parquet / mua_video_spike_count_df.parquet
    paths = sorted(root.rglob("*_video_spike_count_df.parquet"))
    session_map = {}
    for p in paths:
        # Robust session-id inference for both layouts:
        # 1) .../<session>/processed_data/*_video_spike_count_df.parquet
        # 2) .../<session>/*_video_spike_count_df.parquet
        if p.parent.name == "processed_data":
            session_id = p.parent.parent.name
        else:
            session_id = p.parent.name
        # keep first seen file per session by default
        session_map.setdefault(session_id, p)
    return session_map


def apply_condition_filter(df: pl.DataFrame, condition: str) -> pl.DataFrame:
    out = df

    # condition mapping used in efizz barrier datasets
    if USE_CONDITION_FILTER and condition:
        c = str(condition).lower()
        if "barrier_pre_flip" in c:
            if "barrier_present" in out.columns:
                out = out.filter(pl.col("barrier_present") == True)
            if "barrier_flipped" in out.columns:
                out = out.filter(pl.col("barrier_flipped") == False)
        elif "barrier_post_flip" in c:
            if "barrier_present" in out.columns:
                out = out.filter(pl.col("barrier_present") == True)
            if "barrier_flipped" in out.columns:
                out = out.filter(pl.col("barrier_flipped") == True)
        elif "all_time" in c:
            pass

    if REQUIRE_OUT_OF_SHELTER and "OutofshelterIdx" in out.columns:
        out = out.filter(pl.col("OutofshelterIdx") == True)

    if EXCLUDE_ESCAPE and "EscapePeriod" in out.columns:
        out = out.filter(pl.col("EscapePeriod") == False)

    if EXCLUDE_HOMINGS and "homingPeriod" in out.columns:
        out = out.filter(pl.col("homingPeriod") == False)

    return out


def compute_cell_heatmap(df: pl.DataFrame, cluster_id: int, nbins: int = 30):
    needed = ["frames", "spike_clusters", "mouse_x_position", "mouse_y_position", "spike_count"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns in parquet data: {missing}")

    # occupancy from unique frame positions (not per-cluster repeated rows)
    occ_base = (
        df.select(["frames", "mouse_x_position", "mouse_y_position"]) 
          .drop_nulls()
          .unique(subset=["frames"], keep="first")
          .to_pandas()
    )
    if occ_base.empty:
        return None

    # spikes for one cluster
    clu = (
        df.filter(pl.col("spike_clusters") == int(cluster_id))
          .select(["frames", "mouse_x_position", "mouse_y_position", "spike_count"])
          .drop_nulls(["mouse_x_position", "mouse_y_position"])
          .to_pandas()
    )
    if clu.empty:
        return None

    # Ensure numeric
    for col in ["mouse_x_position", "mouse_y_position", "spike_count"]:
        clu[col] = pd.to_numeric(clu[col], errors="coerce")
    clu = clu.dropna(subset=["mouse_x_position", "mouse_y_position", "spike_count"])
    if clu.empty:
        return None

    # shared x/y bins based on occupancy coverage
    x_edges = np.linspace(occ_base["mouse_x_position"].min(), occ_base["mouse_x_position"].max(), nbins + 1)
    y_edges = np.linspace(occ_base["mouse_y_position"].min(), occ_base["mouse_y_position"].max(), nbins + 1)

    occ_base["x_bin"] = pd.cut(occ_base["mouse_x_position"], bins=x_edges, labels=False, include_lowest=True)
    occ_base["y_bin"] = pd.cut(occ_base["mouse_y_position"], bins=y_edges, labels=False, include_lowest=True)
    occ_counts = occ_base.groupby(["y_bin", "x_bin"], dropna=True).size().rename("occ_frames").reset_index()

    clu["x_bin"] = pd.cut(clu["mouse_x_position"], bins=x_edges, labels=False, include_lowest=True)
    clu["y_bin"] = pd.cut(clu["mouse_y_position"], bins=y_edges, labels=False, include_lowest=True)
    spike_sum = clu.groupby(["y_bin", "x_bin"], dropna=True)["spike_count"].sum().rename("spikes").reset_index()

    merged = spike_sum.merge(occ_counts, on=["y_bin", "x_bin"], how="left")
    merged["occ_frames"] = merged["occ_frames"].fillna(0)

    # convert to spikes/s (40 fps in pipeline code)
    fps = 40.0
    merged["rate_hz"] = np.where(merged["occ_frames"] > 0, merged["spikes"] / merged["occ_frames"] * fps, np.nan)

    pivot = merged.pivot(index="y_bin", columns="x_bin", values="rate_hz")

    # reindex full grid so all cells align
    y_idx = np.arange(nbins)
    x_idx = np.arange(nbins)
    pivot = pivot.reindex(index=y_idx, columns=x_idx)

    # Flip vertically to match arena orientation used elsewhere
    return np.flipud(pivot.to_numpy(dtype=float))


def plot_cell_heatmaps_grid(cell_records: pd.DataFrame, session_df: pl.DataFrame, out_dir: Path, session_id: str, tuned_label: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    n_cells = len(cell_records)
    if n_cells == 0:
        return []

    saved = []
    pages = math.ceil(n_cells / MAX_CELLS_PER_PAGE)

    for page in range(pages):
        start = page * MAX_CELLS_PER_PAGE
        stop = min((page + 1) * MAX_CELLS_PER_PAGE, n_cells)
        chunk = cell_records.iloc[start:stop]

        fig, axes = plt.subplots(NROWS, NCOLS, figsize=(3.2 * NCOLS, 3.2 * NROWS), squeeze=False)
        axes = axes.ravel()

        vmax_all = []
        maps = []
        titles = []

        # compute maps first so color scale can be shared
        for _, row in chunk.iterrows():
            cid = int(row["cluster_id"])
            cond = str(row["condition"])
            filt = apply_condition_filter(session_df, cond)
            arr = compute_cell_heatmap(filt, cid, nbins=NBINS)
            maps.append(arr)
            titles.append(f"clu {cid} | {cond}")
            if arr is not None and np.isfinite(arr).any():
                vmax_all.append(np.nanpercentile(arr, 99))

        vmax = max(vmax_all) if vmax_all else 1.0
        vmax = max(vmax, 1e-6)

        for ax, arr, title in zip(axes, maps, titles):
            if arr is None or not np.isfinite(arr).any():
                ax.text(0.5, 0.5, "No spikes/data", ha="center", va="center", transform=ax.transAxes)
                ax.set_title(title, fontsize=8)
                ax.set_xticks([])
                ax.set_yticks([])
                continue

            im = ax.imshow(arr, cmap="viridis", vmin=0, vmax=vmax, interpolation="nearest", aspect="equal")
            ax.set_title(title, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])

        # hide unused axes
        for ax in axes[len(chunk):]:
            ax.axis("off")

        # one shared colorbar (only if at least one valid heatmap exists)
        valid_maps = [m for m in maps if m is not None and np.isfinite(m).any()]
        if valid_maps:
            cbar = fig.colorbar(im, ax=axes[:len(chunk)], fraction=0.02, pad=0.01)
            cbar.set_label("Firing rate (spikes/s)")

        fig.suptitle(f"Spatial spike heatmaps | {tuned_label} | {session_id} | page {page+1}/{pages}", y=1.02)
        fig.tight_layout()

        out_path = out_dir / f"spatial_spike_heatmaps_{tuned_label}_{session_id}_page{page+1:02d}.png"
        fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
        plt.close(fig)
        saved.append(out_path)

    return saved



def preview_session_keys(session_map: dict, n: int = 20):
    keys = sorted(session_map.keys())
    print(f"Example discovered session keys (first {min(n, len(keys))}):")
    for k in keys[:n]:
        print(" -", k)


In [ ]:
# -------------------------
# Load tuned targets + discover session files
# -------------------------

targets = load_tuned_targets(TUNED_IDS_LONG)
session_to_parquet = find_session_parquet_files(PROCESSED_SEARCH_ROOT)

print(f"Loaded tuned target rows: {len(targets):,}")
print("Tuned label counts:")
print(targets["tuned_label"].value_counts(dropna=False).to_string())
print(f"Discovered session parquet files: {len(session_to_parquet):,}")
preview_session_keys(session_to_parquet, n=25)

targets.head()


In [ ]:
# -------------------------
# Resolve session naming mismatch (short labels vs long folder names)
# -------------------------

import re

MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}


def _short_month(mm: int) -> str:
    arr = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
    return arr[mm - 1]


def _mouse_num(name: str):
    m = re.search(r"JAL0*(\d+)", name, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.match(r"0*(\d+)_", name)
    if m:
        return int(m.group(1))
    return None


def _flip_num(name: str):
    m = re.search(r"flip[_ ]?(\d+)", name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def _day_month_from_target(name: str):
    # Matches: 21stSept, 3rdSept, 1apr, 14may
    m = re.search(r"(\d{1,2})(?:st|nd|rd|th)?([A-Za-z]{3,9})", name, flags=re.IGNORECASE)
    if not m:
        return None
    dd = int(m.group(1))
    mon_raw = m.group(2).lower()
    if mon_raw not in MONTHS:
        return None
    mm = MONTHS[mon_raw]
    return (dd, mm)


def _day_month_from_discovered(name: str):
    # Prefer explicit YYYY_MM_DD in folder names
    m = re.search(r"(20\d{2})_(\d{2})_(\d{2})", name)
    if m:
        mm = int(m.group(2))
        dd = int(m.group(3))
        return (dd, mm)

    # Fallback to text token like 1Sept
    return _day_month_from_target(name)


def _target_alias_keys(name: str):
    mouse = _mouse_num(name)
    flip = _flip_num(name)
    dm = _day_month_from_target(name)
    keys = []
    if mouse is None:
        return keys

    if flip is not None and dm is not None:
        keys.append(f"m{mouse}|f{flip}|d{dm[0]}{_short_month(dm[1])}")
    if dm is not None:
        keys.append(f"m{mouse}|d{dm[0]}{_short_month(dm[1])}")
    if flip is not None:
        keys.append(f"m{mouse}|f{flip}")
    keys.append(f"m{mouse}|raw:{name.lower()}")
    return keys


def _discovered_alias_keys(name: str):
    mouse = _mouse_num(name)
    flip = _flip_num(name)
    dm = _day_month_from_discovered(name)
    keys = []
    if mouse is None:
        return keys

    if flip is not None and dm is not None:
        keys.append(f"m{mouse}|f{flip}|d{dm[0]}{_short_month(dm[1])}")
    if dm is not None:
        keys.append(f"m{mouse}|d{dm[0]}{_short_month(dm[1])}")
    if flip is not None:
        keys.append(f"m{mouse}|f{flip}")
    keys.append(f"m{mouse}|raw:{name.lower()}")
    return keys


def resolve_session_names(target_sessions, discovered_session_names):
    key_to_discovered = {}
    for ds in discovered_session_names:
        for k in _discovered_alias_keys(ds):
            key_to_discovered.setdefault(k, []).append(ds)

    mapping = {}
    unresolved = []
    ambiguous = {}

    for ts in sorted(set(target_sessions)):
        candidates = []
        # strong -> weaker keys in this order
        for k in _target_alias_keys(ts):
            hits = key_to_discovered.get(k, [])
            if len(hits) == 1:
                mapping[ts] = hits[0]
                candidates = []
                break
            if len(hits) > 1:
                candidates = hits
                # keep checking for a stricter unique match; if none, mark ambiguous later
        else:
            if candidates:
                ambiguous[ts] = sorted(set(candidates))
            else:
                unresolved.append(ts)

    return mapping, unresolved, ambiguous


resolved_map, unresolved, ambiguous = resolve_session_names(
    target_sessions=targets["session"].tolist(),
    discovered_session_names=list(session_to_parquet.keys()),
)

targets = targets.copy()
targets["session_discovered"] = targets["session"].map(resolved_map)
keep = targets.dropna(subset=["session_discovered"]).copy()
keep["session_discovered"] = keep["session_discovered"].astype(str)

missing = sorted(set(targets["session"]) - set(keep["session"].unique()))

print(f"Resolved target rows: {len(keep):,} / {len(targets):,}")
print("Sanity check: session_discovered column added to targets")
print(f"Resolved sessions: {keep['session'].nunique():,} / {targets['session'].nunique():,}")
print(f"Unresolved sessions: {len(unresolved):,}")
print(f"Ambiguous sessions: {len(ambiguous):,}")

if unresolved:
    print("First unresolved sessions:")
    print(unresolved[:20])

if ambiguous:
    print("First ambiguous sessions (target -> options):")
    for i, (k, v) in enumerate(sorted(ambiguous.items())):
        if i >= 10:
            break
        print(f"  {k} -> {v}")

keep.groupby(["tuned_label", "session"]).size().reset_index(name="n_cells").head(20)


Resolved target rows: 0 / 410
Resolved sessions: 0 / 22
Unresolved sessions: 22
Ambiguous sessions: 0
First unresolved sessions:
['JAL005_21stSept', 'JAL005_8thSept', 'JAL3_1sept', 'JAL3_25aug', 'JAL3_4sept', 'JAL3_7sept', 'JAL4_11thSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_3rdSept', 'JAL6_28mar', 'JAL6_flip5_25mar', 'JAL6_flip7_1apr', 'JAL7_23apr', 'JAL7_flip2_12mar', 'JAL7_flip5_22mar', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL8_14may', 'JAL8_flip1_25apr']


,tuned_label,session,n_cells


In [ ]:
# -------------------------
# Match targets to discovered sessions (uses alias-resolved names)
# -------------------------

if "session_discovered" not in targets.columns:
    if "resolve_session_names" not in globals():
        raise RuntimeError("Resolver functions are not loaded. Run the session-name resolver cell first.")
    resolved_map, unresolved, ambiguous = resolve_session_names(
        target_sessions=targets["session"].tolist(),
        discovered_session_names=list(session_to_parquet.keys()),
    )
    targets = targets.copy()
    targets["session_discovered"] = targets["session"].map(resolved_map)

keep = targets.dropna(subset=["session_discovered"]).copy()
keep["session_discovered"] = keep["session_discovered"].astype(str)

missing = sorted(set(targets["session"]) - set(keep["session"].unique()))

print(f"Matched target rows: {len(keep):,} / {len(targets):,}")
print(f"Matched sessions: {keep['session'].nunique():,} / {targets['session'].nunique():,}")
print(f"Missing sessions: {len(missing):,}")
if missing:
    print("First missing sessions:")
    print(missing[:20])

keep.groupby(["tuned_label", "session"]).size().reset_index(name="n_cells").head(20)


Matched target rows: 0 / 410
Matched sessions: 0 / 22
Missing sessions: 22
First missing sessions:
['JAL005_21stSept', 'JAL005_8thSept', 'JAL3_1sept', 'JAL3_25aug', 'JAL3_4sept', 'JAL3_7sept', 'JAL4_11thSept', 'JAL4_19thSept', 'JAL4_28aug', 'JAL4_3rdSept', 'JAL6_28mar', 'JAL6_flip5_25mar', 'JAL6_flip7_1apr', 'JAL7_23apr', 'JAL7_flip2_12mar', 'JAL7_flip5_22mar', 'JAL7_sesh8_9apr', 'JAL7_sesh9_16apr', 'JAL8_14may', 'JAL8_flip1_25apr']


,tuned_label,session,n_cells


In [ ]:
# -------------------------
# Build and save heatmap figures (many cells per page)
# -------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
all_saved = []

# group by tuned label and session so each figure remains interpretable
for (lab, sess), grp in keep.groupby(["tuned_label", "session"], sort=True):
    sess_key = grp["session_discovered"].iloc[0] if "session_discovered" in grp.columns else sess
    parquet_path = session_to_parquet.get(sess_key)
    if parquet_path is None:
        continue

    print(f"\n[{lab}] session={sess} | cells={len(grp)}")
    print(f"  target session: {sess}")
    print(f"  discovered session: {sess_key}")
    print(f"  reading: {parquet_path}")

    session_df = pl.read_parquet(parquet_path, low_memory=True, use_pyarrow=True, memory_map=True)

    saved_paths = plot_cell_heatmaps_grid(
        cell_records=grp,
        session_df=session_df,
        out_dir=OUT_DIR,
        session_id=sess,
        tuned_label=lab,
    )
    all_saved.extend(saved_paths)
    print(f"  saved pages: {len(saved_paths)}")

print(f"\nTotal pages saved: {len(all_saved)}")
if all_saved:
    print("Example outputs:")
    for p in all_saved[:10]:
        print(f" - {p}")



Total pages saved: 0


## Notes
- Heatmap values are **spikes/s**, normalized by occupancy per position bin.
- Occupancy is computed from unique frames (`frames`) in the same condition mask.
- Figures are saved as `spatial_spike_heatmaps_{tuned_label}_{session}_pageXX.png`.
- Increase/decrease `NROWS`, `NCOLS`, `NBINS`, and `MAX_CELLS_PER_LABEL` to control output size and runtime.
